In [0]:
## read raw data into dataframe

customers = spark.read.table("db_academy.pii_data.customers_raw")

print(f"number of customer: {customers.count()}")
display(customers.limit(10))

In [0]:
import uuid
from pyspark.sql.functions import udf

customers_raw = spark.read.table("db_academy.pii_data.customers_raw")

# Create udf to generate unique token
def generate_token():
    return(str(uuid.uuid4()))

generate_token_udf = udf(generate_token)

# Get unique email and generate token for each email
email_lkp_token = (
    customers_raw.select("email").distinct()
    .withColumn("email_token", generate_token_udf())
)

# write df to table
email_lkp_token.write.mode("overwrite").saveAsTable("db_academy.pii_data.email_lkp_token")

print(f"Unique emails: {email_lkp_token.count()}")
display(email_lkp_token.limit(10))


In [0]:
from pyspark.sql.functions import col

# Join raw customer to email_lkp_token to get token for email
customers_tokenized = (
    customers_raw.alias("a").join(email_lkp_token.alias("b"), on="email", how="left")
    .select(
        "customer_id",
        "full_name",
        col("a.email").alias("email_original"),
        col("email_token").alias("email"),
        "phone",
        "ssn",
        "date_of_birth",
        "address",
        "city",
        "country",
    )
)

customers_tokenized.write.mode("overwrite").saveAsTable("db_academy.pii_data.customers_tokenized")

print(f"number of customer: {customers_tokenized.count()}")
display(customers_tokenized.limit(10))
#